In [1]:
import torch
import pandas as pd
import numpy as np
import re
import evaluate
import nltk
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import LEDTokenizer, LEDForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
# 1. Load the pre-mixed dataset
print("Loading pre-mixed dataset...")
df_mixed = pd.read_csv("mixed_meeting_dataset.csv")

def clean_text(text):
    if not isinstance(text, str): return str(text)
    text = re.sub(r'#(Person\d+)#:', r'\1:', text)
    text = re.sub(r'#(Person\d+)#',  r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_mixed['Transcript'] = df_mixed['Transcript'].apply(clean_text)
df_mixed['Summary'] = df_mixed['Summary'].apply(clean_text)

print(f"Total Mixed Dataset Size: {len(df_mixed)}")
print(df_mixed.head())

full_dataset = Dataset.from_pandas(df_mixed)

# split 80:20
split_dataset = full_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")

Loading pre-mixed dataset...
Total Mixed Dataset Size: 1460
                                          Transcript  \
0  Speaker 3: increases applicable. To certain re...   
1  Speaker 0: Okay. Our last agenda item is the m...   
2  #Person1#: I hear you are moving to Dalian. #P...   
3  #Person1#: Would you like to see our new shirt...   
4  Speaker 1: Please read item 31. Speaker 7: The...   

                                             Summary  
0  Adoption of Resolution Calling an Election to ...  
1  A MOTION requesting the executive establish an...  
2  #Person2#'s moving to Dalian for work. #Person...  
3  #Person1# tries to sell the new shirts to #Per...  
4  AN ORDINANCE authorizing the Director of Seatt...  
Train size: 1168, Eval size: 292


In [ ]:
# Tokenization
model_checkpoint = "MingZhong/DialogLED-base-16384"
tokenizer = LEDTokenizer.from_pretrained(model_checkpoint)

MAX_INPUT_LENGTH = 4096 
MAX_TARGET_LENGTH = 256

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["Transcript"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        text_target=examples["Summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    
    # Global Attention Mask
    global_attention_mask = []
    for seq in model_inputs["input_ids"]:
        mask = [0] * len(seq)
        mask[0] = 1 
        non_pad_indices = [i for i, token in enumerate(seq) if token != tokenizer.pad_token_id]
        if non_pad_indices:
            mask[non_pad_indices[-1]] = 1 
        global_attention_mask.append(mask)
        
    model_inputs["global_attention_mask"] = global_attention_mask
    return model_inputs

print("Tokenizing training dataset...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)

print("Tokenizing eval dataset...")
tokenized_eval = eval_dataset.map(preprocess_function, batched=True, remove_columns=eval_dataset.column_names)

Tokenizing training dataset...


Map:   0%|          | 0/1168 [00:00<?, ? examples/s]

Tokenizing eval dataset...


Map:   0%|          | 0/292 [00:00<?, ? examples/s]

In [ ]:
# Training
model = LEDForConditionalGeneration.from_pretrained(
    model_checkpoint,
    gradient_checkpointing=True, # Important to save memory for LED
    use_safetensors=True
).to(DEVICE)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Metrics
rouge = evaluate.load("rouge")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]
    
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)
    
    return {k: round(v, 4) for k, v in result.items()}

training_args = Seq2SeqTrainingArguments(
    output_dir="./led_mixed_output",
    learning_rate=1e-5,
    per_device_train_batch_size=1, # LED is heavy
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    weight_decay=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True if torch.cuda.is_available() else False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Starting training...")

trainer.train()

model.safetensors:   0%|          | 0.00/648M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/296 [00:00<?, ?it/s]

Starting training...


Epoch,Training Loss,Validation Loss
1,No log,0.907376
2,No log,0.631387
3,No log,0.609511


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=438, training_loss=14.70613272331621, metrics={'train_runtime': 3659.6819, 'train_samples_per_second': 0.957, 'train_steps_per_second': 0.12, 'total_flos': 9461517741195264.0, 'train_loss': 14.70613272331621, 'epoch': 3.0})

In [5]:
# Save Model
save_path = "./led_mixed_finetuned"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./led_mixed_finetuned


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

print("final eval")

trainer.compute_metrics = compute_metrics

eval_results = trainer.evaluate()
print("\n=== EVALUATION RESULTS ===")
for key, value in eval_results.items():
    print(f"{key}: {value}")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jerik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jerik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Starting final evaluation...


Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
No log,0.609511,3,0.451300,0.278300,0.371100,0.414600,73.359600



=== EVALUATION RESULTS ===
eval_loss: 0.6095112562179565
eval_rouge1: 0.4513
eval_rouge2: 0.2783
eval_rougeL: 0.3711
eval_rougeLsum: 0.4146
eval_gen_len: 73.3596
